In [0]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp

# --- TASK 1: CLEAN SESSIONS ---
bronze_sessions_df = spark.read.table("f1_project.bronze.raw_sessions_2025")

silver_sessions_df = (bronze_sessions_df
    .withColumn("start_time", to_timestamp(col("date_start")))
    .withColumn("end_time", to_timestamp(col("date_end")))
    .filter(col("session_type").isin("Race", "Qualifying"))
    .select(
        col("session_key").alias("session_id"),
        "session_name",
        "session_type",
        "location",
        "country_name",
        "start_time",
        "end_time",
        col("year").cast("int")
    )
    .withColumn("processed_at", current_timestamp())
)

silver_sessions_df.write.format("delta").mode("overwrite").saveAsTable("f1_project.silver.cleaned_sessions")

# --- TASK 2: ENRICH LAPS WITH DRIVER NAMES ---
# This is where the real Data Engineering happens!
bronze_laps = spark.read.table("f1_project.bronze.raw_laps_2025")
bronze_drivers = spark.read.table("f1_project.bronze.raw_drivers_2025")

silver_laps_enriched = (bronze_laps
    .join(bronze_drivers.select("driver_number", "full_name", "team_name"), on="driver_number", how="inner")
    .select(
        "full_name",
        "team_name",
        col("lap_number").cast("int"),
        col("lap_duration").cast("float"),
        col("is_pit_out_lap").cast("boolean"),
        current_timestamp().alias("processed_at")
    )
)

silver_laps_enriched.write.format("delta").mode("overwrite").saveAsTable("f1_project.silver.enriched_laps")

print("Silver Transformation for Sessions and Enriched Laps Complete!")